In [ ]:
import requests, sqlite3, pandas as pd

url = "https://raw.githubusercontent.com/davidjamesknight/SQLite_databases_for_learning_data_science/main/diamonds.db"
r = requests.get(url)

with open("diamonds.db", "wb") as f:
    f.write(r.content)

conn = sqlite3.connect("diamonds.db")

query = """
SELECT
  O.carat,
  O.price,
  O.depth,
  "O"."table",
  O.x,
  O.y,
  O.z,
  C.cut,
  Co.color,
  Cl.clarity
FROM
  Observation AS O
JOIN
  Cut AS C ON O.cut_id = C.cut_id
JOIN
  Color AS Co ON O.color_id = Co.color_id
JOIN
  Clarity AS Cl ON O.clarity_id = Cl.clarity_id
"""

df = pd.read_sql_query(query, conn)
df.head()

## 📋 Recap del Análisis Exploratorio

En el notebook anterior exploramos el dataset de **53,940 diamantes**. Aquí un resumen rápido:

### Variables
| Tipo | Variables |
|---|---|
| **Numéricas** | `carat`, `depth`, `table`, `x`, `y`, `z`, `price` |
| **Categóricas (ordinales)** | `cut` (Fair → Premium), `color` (D → J), `clarity` (IF → I1) |

### Hallazgos clave
- 💎 **`carat`**, **`x`**, **`y`** y **`z`** tienen la correlación más fuerte con `price`
- ⚠️ `depth` y `table` muestran alta dispersión y outliers elevados
- 📊 `price` y `carat` tienen distribuciones **sesgadas a la derecha**
- 🔗 Las variables de dimensión (`x`, `y`, `z`) están **altamente correlacionadas entre sí** → posible multicolinealidad

### Variable objetivo
> Predeciremos **`price`** (precio en USD) usando las demás características del diamante.

## 1️⃣ Preprocesamiento

Antes de ajustar el modelo, preparamos los datos:
- Separamos la variable objetivo (`price`) de las variables predictoras
- Dividimos en **80% entrenamiento / 20% prueba**

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

### Encoding de variables categóricas

Las variables `cut`, `color` y `clarity` son categóricas ordinales.  
Usamos **OrdinalEncoder** respetando el orden natural de cada una.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Orden natural de cada variable categórica ordinal
cut_order      = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
color_order    = ['J', 'I', 'H', 'G', 'F', 'E', 'D']   # D es el mejor
clarity_order  = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

enc = OrdinalEncoder(
    categories=[
        cut_order, 
        color_order, 
        clarity_order
    ]
)

cat_cols = ['cut', 'color', 'clarity']

# Fit en train, transform en ambos
X_train_enc = X_train.copy()
X_test_enc  = X_test.copy()

X_train_enc[cat_cols] = enc.fit_transform(X_train[cat_cols])
X_test_enc[cat_cols]  = enc.transform(X_test[cat_cols])

X_train_enc.head()

## 2️⃣ Ajuste del Modelo — Regresión Lineal (OLS)

Ajustamos un modelo **crudo** con todas las variables, sin ningún tipo de selección.  
Usamos `statsmodels` para ver el resumen estadístico completo.

In [ ]:
import statsmodels.api as sm

X_train_const = sm.add_constant(X_train_enc)

model_sm = sm.OLS(y_train, X_train_const)
results  = model_sm.fit()

print(results.summary())

## 3️⃣ Evaluación en Test

Medimos el desempeño del modelo con datos que **nunca vio durante el entrenamiento**.

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
from math import sqrt

X_test_const = sm.add_constant(X_test_enc)
y_pred = results.predict(X_test_const)

r2   = r2_score(y_test, y_pred)
rmse = sqrt(mean_squared_error(y_test, y_pred))

print(f"R²   en test: {r2:.4f}")
print(f"RMSE en test: {rmse:.2f} USD")